# 04 · `gl_engine/erc/discovery.py`

## What this file is for

Find every package ISO gave us, and ask each one who it is.

Two rules govern this file and both were learned the hard way. **Identity comes from the namespace declared inside the package, never from the folder name** — the corpus is unpacked inconsistently, some packages sit inside a wrapper directory, and folder names use spaces where the namespace uses underscores. A reader keyed on the path will disagree with ISO about what it is holding.

And **a search predicate must never define a population**: discovery takes every directory containing a `DataDefs/`, rather than matching a naming pattern, because a package with an unusual folder name is still a package.

**Depends on:** [`01-config`](01-config.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.erc import discovery

for name, obj in vars(discovery).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != discovery.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Walk the corpus and count what's there.

In [ ]:
from gl_engine.erc.discovery import discover

packages = discover()
print("packages discovered :", len(packages))

jurisdictions = {p.identity.juris for p in packages}
print("jurisdictions       :", len(jurisdictions))
print("countrywide packages:", sum(1 for p in packages if p.is_countrywide))

## The interesting case

### The folder name and the identity are not the same string

This is the trap the module exists to avoid. Compare what the directory is called with what the package says it is.

In [ ]:
for p in packages[:6]:
    folder = p.content.name
    mark = "  <-- differs" if folder != p.pkg_id else ""
    print(f"folder   {folder}")
    print(f"identity {p.pkg_id}{mark}\n")

### An identity is structured, and that structure is what makes editions sortable

In [ ]:
p = packages[0]
i = p.identity
print("pkg_id  :", p.pkg_id)
print("juris   :", i.juris)
print("edition :", i.edition)
print("version :", i.version)
print("sort_key:", i.sort_key)
print()
print("declared parent:", p.declared_parent)
print("content dir    :", p.content.name)

`sort_key` is why `(edition, version)` ordering works everywhere else — same-day filings exist, and version breaks the tie. [`05-resolve-resolver`](05-resolve-resolver.ipynb) depends entirely on this being right.

## What it refuses

A package that cannot say who it is, is not silently skipped.

In [ ]:
from gl_engine.errors import IdentityError

print(IdentityError.__doc__ or "(no docstring)")
print()
print("Every discovered package resolved an identity:",
      all(p.pkg_id for p in packages))
print("None fell back to its folder name:",
      not any(p.pkg_id == p.content.name for p in packages))

## Try it yourself

1. How many packages live inside a `_MachineReadableContent` wrapper rather than directly in the jurisdiction folder?
2. Group the packages by jurisdiction. Which jurisdiction has the most editions, and how many?
3. `xsd_count` is on every package. Find one with more than one, and work out why.

In [ ]:
# your turn